<a href="https://colab.research.google.com/github/DABMASTER-Brought-me-into-this/ZeroToHeroColabCollection/blob/main/recall_makemore_pt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install matplotlib

In [ ]:
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F
%matplotlib inline

In [ ]:
# Retrieving Data
words = open('data.txt').read().split()

In [ ]:
# Assigning Each Character Value A Int Val
stoi = {char: ord(char) - 96 for char in sorted(list(set(''.join(words))))}
stoi['.'] = 0
itos = {val: key for key, val in stoi.items()}

### BIGRAM RECALL

In [ ]:
# Counts Tensor
N = torch.zeros((27,27), dtype=torch.int64)


# Assigning Counts to Counts Tensor
for word in words:
  chs = ['.'] + list(word) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    N[ix1, ix2] += 1

In [ ]:
# Attempting To Draw It
def draw_fig(N):
  plt.figure(figsize = (16,16))
  plt.imshow(N, cmap="Blues")

  for i in range(N.shape[0]):
    for j in range(N.shape[1]):
      plt.text(j, i, itos[j] + itos[i], ha = "center", va = "top")
      plt.text(j, i, str(N[j, i].item()), ha = "center", va = "bottom")

In [ ]:
# Creating Probability Tensor
P = N.float() / N.float().sum(1, keepdim = True)

In [ ]:
# Building Names Now
for x in range(10):
  chars = ['.']
  n = 0
  while True:
    chars.append(itos[torch.multinomial(P[stoi[chars[len(chars) - 1]]], 1, False).item()])
    if chars.count('.') >= 2:
      break
  print(''.join(char if char != '.' else '' for char in chars).capitalize())

Ryn
Kslyabe
Abiazerdihynn
Kyday
Marela
Ynnaililinadilb
Leasinn
Fizalazakercienin
Leth
Annn


In [ ]:
log_likelihood = 0.0
n = 0

for word in words:
# for word in ['tanisha']:
  chs = ['.'] + list(word.lower()) + ['.'] # Convert word to lowercase here
  for letter_index in range(len(chs)-1):
    ix1 = stoi[chs[letter_index]]
    ix2 = stoi[chs[letter_index + 1]]
    prob = P[ix1, ix2]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1

print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n=}')

log_likelihood=tensor(-559891.7500)
nll=tensor(559891.7500)
nll/n=tensor(2.4541)


### NEURAL NETWORK RECALL

In [ ]:
# Inputs & Outputs
xs, ys = [], []

# Adding Inputs & Outputs
for word in words:
  chs = ['.'] + list(word) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    xs.append(stoi[ch1])
    ys.append(stoi[ch2])

# Converting Inputs & Outputs to Tensors
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()


# Weights
W = torch.randn((27, 27), requires_grad = True)

In [ ]:
for k in range(100):
  # Forward Pass
  xenc = F.one_hot(xs, 27).float() # Encoded Xs
  logits = xenc @ W # Log Counts
  # --- Soft Max Start ---
  counts = logits.exp() # Predicted Counts
  prob = counts / counts.sum(1, keepdim = True) # Probabilities
  # --- Soft Max End ---
  loss = -prob[torch.arange(0,num), ys].log().mean()

  # Backwards Pass
  W.grad = None
  loss.backward()

  # Update
  W.data -= 50 * W.grad
  print(k, loss)

print(f"Ideal Loss: {nll/n}")
print(f"Actual Loss: {-prob[torch.arange(0,num), ys].log().mean()}")

0 tensor(2.4732, grad_fn=<NegBackward0>)
1 tensor(2.4730, grad_fn=<NegBackward0>)
2 tensor(2.4728, grad_fn=<NegBackward0>)
3 tensor(2.4726, grad_fn=<NegBackward0>)
4 tensor(2.4724, grad_fn=<NegBackward0>)
5 tensor(2.4722, grad_fn=<NegBackward0>)
6 tensor(2.4720, grad_fn=<NegBackward0>)
7 tensor(2.4718, grad_fn=<NegBackward0>)
8 tensor(2.4716, grad_fn=<NegBackward0>)
9 tensor(2.4714, grad_fn=<NegBackward0>)
10 tensor(2.4712, grad_fn=<NegBackward0>)
11 tensor(2.4710, grad_fn=<NegBackward0>)
12 tensor(2.4709, grad_fn=<NegBackward0>)
13 tensor(2.4707, grad_fn=<NegBackward0>)
14 tensor(2.4705, grad_fn=<NegBackward0>)
15 tensor(2.4704, grad_fn=<NegBackward0>)
16 tensor(2.4702, grad_fn=<NegBackward0>)
17 tensor(2.4700, grad_fn=<NegBackward0>)
18 tensor(2.4699, grad_fn=<NegBackward0>)
19 tensor(2.4697, grad_fn=<NegBackward0>)
20 tensor(2.4696, grad_fn=<NegBackward0>)
21 tensor(2.4694, grad_fn=<NegBackward0>)
22 tensor(2.4693, grad_fn=<NegBackward0>)
23 tensor(2.4692, grad_fn=<NegBackward0>)
24